# Suite DOM — Value domains

A value domain names a standard's representation, with only its parameters. Domains are data: a standard dialect
(ISO/IEC 11404's, IEEE 754's, IEEE 1164's or Unicode's), a singleton, owns each by name, and each one's schema is a named
`Schemas.OfAny` of the scalar it narrows, related to it both ways; they are stored and round-trip, and the framework's
interface reads them (`kind_of`, `properties`, `name_of`): a domain's kind, by its dialect's tag, and its properties,
its parameters and the ones inferred from them, populated by hand. A language's own kinds are its dialect's too. A term
carries no domain, so evaluation takes values by the schemas they are written with, ints unbounded and floats binary64,
and inference gives the natives' domains, each the values of one native type, by the signatures of Patterns' operations
over them and over collections. IEEE 754's arithmetic, which C's and SystemVerilog's evaluators use, is tested here
directly.

In [ ]:
import decimal
import math
import struct

from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Python import Python3
from mbse.Expressions.Arm import AAPCS64
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.C import Preprocessor as CPreprocessor, Types as CTypes  # the retrofit's new dialects
from mbse.Expressions.Cpp import RegExp as CppRegExp, Types as CppTypes
from mbse.Expressions.POSIX import BRE as PosixBRE, ERE as PosixERE  # POSIX's, which a C++ regex may hold
from mbse.Expressions.Python import Patterns as PythonPatterns, RegExp as PythonRegExp, Types as PythonTypes
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, RegExp as TypeScriptRegExp, Types as TypeScriptTypes
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Patterns.Expressions import Domains as D, Evaluators as V, Expressions as E, Ieee754
from mbse.Expressions.Framework import Domains as FD, Standards as FS, Terms
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

INSTALLS = [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install]
store = Builders.Store()
for install in INSTALLS:  # the framework, the standard dialects, then Patterns: each its own transaction
    store.transaction(install)
store.transaction(Patterns.define)  # Patterns' operations, by name, which validation reads


def term(spec):
    """The term `spec` writes, the root of an expression built in a transaction."""
    with store.transaction():
        return Terms.term(store.Expressions.Framework.Expression(lambda e: e.root(E.DIALECT.spec(spec))))


def val(spec, scope=None):
    """The value of the term `spec` writes."""
    return V.OfAny(term(spec), scope)


def ev(name, *arguments, scope=None):
    """The value of a Patterns call."""
    return val(E.operation(name, *arguments), scope)

<a id="DOM-01"></a>
## DOM-01 · Inference's natives' domains are each the values of one native type; a value's is its type's, or Object

In [ ]:
# The natives' domains, the framework's OfTypes, each named by its type, and a native's domain its type's.
NATIVES = [(D.Bool, "bool", True), (D.Int, "int", 1), (D.Float, "float", 1.5), (D.Str, "str", "a"), (D.Bytes, "bytes", b"a")]
assert all(isinstance(domain, FD.OfTypes) and domain.name() == repr(domain) == name for domain, name, _ in NATIVES)
assert [D.of(value) for _, _, value in NATIVES] == [domain for domain, _, _ in NATIVES]
# Each holds the values of exactly its type: a bool is not an int, nor an int a float; ints are unbounded.
assert D.Int.contains(10 ** 30) and not D.Int.contains(True) and not D.Bool.contains(1) and not D.Float.contains(1)
assert D.Str.contains("日本語 🚀") and not D.Str.contains(b"a") and D.Bytes.contains(b"") and not D.Bytes.contains("abc")
# A domain includes only itself: Patterns has no implied promotions.
assert D.Int.includes(D.Int) and not D.Int.includes(D.Float) and not D.Float.includes(D.Int) and not D.Int.includes(D.Bool)
assert D.Float.includes(FD.OfUnion(D.Float)) and not D.Int.includes(FD.OfUnion(D.Int, D.Float)) and FD.overlaps(FD.Anything, D.Int)
# Any other value of the dialect is a data object, a term among them; None is no value of it.
assert D.of(term(E.variable("x"))) is D.Object and D.Object.name() == "object"
with raises(TypeError, match="a NoneType is not a value of the Patterns dialect"):
    D.of(None)

<a id="DOM-03"></a>
## DOM-03 · Domains are data: a dialect owns each by name, and each one's schema, an OfAny of its scalar, reads it

In [ ]:
DIALECTS = {ISOIEC11404: "ISOIEC11404", IEEE754: "IEEE754", IEEE1164: "IEEE1164", Unicode: "Unicode"}  # each by its family
STANDARD = {  # the standard dialects' domains, each the scalar its schema narrows
    "ISOIEC11404": {"boolean": "bool", "integer": "int", "octetstring": "bytes", "octet": "int", "modulo(256)": "int"},
    "IEEE754": {"binary64": "float", "binary32": "float"}, "IEEE1164": {"std_logic": "str"}, "Unicode": {"string": "str"}}
defined = {family: store.transaction(module.define)[0] for module, family in DIALECTS.items()}  # each dialect's domains
singletons = {family: getattr(store.Instances.Expressions, family).Expressions for family in STANDARD}  # by its path
assert all(singletons[family] is defined[family]["dialect"] for family in STANDARD)
iso = singletons["ISOIEC11404"]
domains = {str(e.name): e.domain for family in STANDARD for e in singletons[family].domains}
assert {family: [str(e.name) for e in singletons[family].domains] for family in STANDARD} == {
    family: list(names) for family, names in STANDARD.items()}
assert iso.domains({"name": "octet"}) is domains["octet"] is defined["ISOIEC11404"]["octet"]
assert store.transaction(ISOIEC11404.define)[0]["octet"] is domains["octet"]  # again: nothing
assert {name: FD.kind_of(domain)[1] for name, domain in domains.items()} == {  # each its kind, read through the interface
    "boolean": "boolean", "integer": "integer", "octetstring": "octetstring", "octet": "octet", "modulo(256)": "modulo",
    "binary64": "ieee754", "binary32": "ieee754", "std_logic": "ieee1164", "string": "unicode"}
SPELLED = {"modulo(256)": "modulo_256"}  # a schema's name is a dotted identifier: 11404's notation spelled as one
for family, names in STANDARD.items():  # each one's schema: an OfAny of its scalar, its domain read both ways
    for name, scalar in names.items():
        schema = store.registered(f"Expressions.{family}.Expressions.{SPELLED.get(name, name)}")
        assert schema.schemas == (store.registered(f"Schemas.Of{scalar.capitalize()}"),) and schema.validate() == []
        assert [e.schema for e in domains[name].schema] == [schema]
        read = [e.domain for e in Builders.OfAny.Builder(schema, False, store).domain]
        assert [(FD.name_of(d), FD.kind_of(d)[:2], FD.properties(d)) for d in read] == [
            (name, FD.kind_of(domains[name])[:2], FD.properties(domains[name]))]
Domain, HasDomain = store.registered("Expressions.Framework.Domain"), store.registered("Expressions.Framework.HasDomain")
assert HasDomain.uniques == (frozenset({"schema"}), frozenset({"domain"})) and Domain.adjacencies["dialect"].owner
# The framework's union of kinds is of the dialects' unions, each named by its dialect's tag.
DomainKind = store.registered("Expressions.Framework.DomainKind")
assert [b.name for b in DomainKind.branches] == ["isoiec11404", "ieee754", "ieee1164", "unicode"]
assert [b.name for b in DomainKind.branches[0].schema.branches] == [
    "boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"]
# A domain of its own, owned by its dialect by name; a packed one links its enum and its representation, domains too.
with store.transaction():
    state = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(
        lambda u: u.isoiec11404(lambda v: v.state(lambda e: e.members(["IDLE", "RUN"]).ordered(False)))))))
    coded = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(lambda u: u.isoiec11404(
        lambda v: v.packed(lambda e: e.codes([0, 1]).ordered(True)))))).packs(lambda a: a.insert(
        lambda e: e.domain(state).representation(domains["octet"]))))
    bad = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(
        lambda u: u.isoiec11404(lambda v: v.integer(lambda e: e.width(0)))))))
    empty = store.Expressions.Framework.Domain(lambda d: d)
    store.Expressions.ISOIEC11404.Expressions.Dialect(iso, lambda b: b.domains({
        "state": lambda e: e.domain(state), "coded": lambda e: e.domain(coded), "bad": lambda e: e.domain(bad),
        "empty": lambda e: e.domain(empty)}))  # each owned: written with its owner
OWN = [*STANDARD["ISOIEC11404"], "state", "coded", "bad", "empty"]
assert [str(e.name) for e in iso.domains] == OWN
assert [FD.kind_of(domain)[1] for domain in (state, coded, bad)] == ["state", "packed", "integer"]
assert [(e.domain, e.representation) for e in coded.packs] == [(state, domains["octet"])]  # packed into octet codes
# A width is data: that it is positive is a constraint for a later resolver, not the store's.
assert FD.properties(bad) == {"width": 0} and FD.name_of(bad) == "bad"
with raises(ValueError, match="a domain needs a kind"):
    FD.kind_of(empty)

<a id="DOM-05"></a>
## DOM-05 · Over the wire a domain is its dialect's, by name: it round-trips, and so does a store holding it

In [ ]:
Dialect = store.registered("Expressions.ISOIEC11404.Expressions.Dialect")
# Each a document of its own, by its path: read back, it is the object the store holds. Its kind is written its
# dialect's, then its kind's.
assert '"isoiec11404": {"octet": {"ordered": true}}' in JSON.ToJSON(store)(Domain, domains["octet"])
assert ('"isoiec11404": {"modulo": {"modulus": 256, "least": 0, "greatest": 255, "ordered": true}}' in
        JSON.ToJSON(store)(Domain, domains["modulo(256)"]))
assert '"ieee754": {"ieee754": {"format": "binary32", "rounding": "roundTiesToEven",' in JSON.ToJSON(store)(Domain, domains["binary32"])
for domain in [domains["octet"], domains["modulo(256)"], domains["binary32"], state, coded]:
    for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
        back = decode(Domain, encode(Domain, domain))
        assert back is domain and FD.properties(back) == FD.properties(domain)  # owned: its path, its dialect's and its name, finds it
back = JSON.FromJSON(store)(Dialect, JSON.ToJSON(store)(Dialect, iso))
assert back is iso and [str(e.name) for e in back.domains] == OWN
# A store holding them round-trips whole, after its documents were read back, into a store with the same installs: nothing lost, nothing written twice.
snapshot = JSON.ToJSON(store).Snapshot()
again = Builders.Store()
for install in INSTALLS:
    again.transaction(install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot
restored = {str(e.name): e.domain for e in again.Instances.Expressions.ISOIEC11404.Expressions.domains}  # keyed by name
assert sorted(restored) == sorted(OWN)
assert [(FD.kind_of(restored[name])[1], FD.properties(restored[name])) for name in ("state", "coded", "bad")] == [
    ("state", {"members": ("IDLE", "RUN"), "ordered": False}), ("packed", {"codes": (0, 1), "ordered": True}),
    ("integer", {"width": 0})]
assert [(e.domain, e.representation) for e in restored["coded"].packs] == [(restored["state"], restored["octet"])]
# A term carries no domain: a literal holds a value, its spelling and its problems, and its value's schema is its own.
assert list(store.registered("Expressions.Framework.Literal").properties) == ["value", "text", "problems"]
with raises(TypeError):
    E.literal(200, "uint8")

<a id="DOM-06"></a>
## DOM-06 · Evaluation compares values by their schemas: of one scalar, else incomparable, and floats as mbse-schemas does

In [ ]:
# Evaluation compares values by the schemas they are written with: of one scalar, else incomparable.
assert ev("lt", -1, 1) is True and ev("eq", 5, 5.0) is None and ev("eq", "RUN", "RUN") is True
assert ev("lt", b"\x00\x01", b"\x01\x00") is True  # bytes by their octets
# Floats compare as mbse-schemas compares floats: NaNs equal each other, and -0 comes before 0.
nan = float("nan")
assert ev("eq", nan, nan) is True and ev("lt", nan, 1.0) is None and ev("lt", -0.0, 0.0) is True

<a id="DOM-07"></a>
## DOM-07 · Evaluation's ints are unbounded and its floats binary64, and it does not promote

In [ ]:
# Evaluation takes ints by their schema, unbounded, and floats as binary64; it does not promote.
assert ev("add", 2 ** 70, 1) == 2 ** 70 + 1 and ev("neg", -128) == 128 and ev("mul", 1.5, 2.0) == 3.0
with raises(TypeError, match="add expects numbers of one domain, got int and float"):
    ev("add", 1, 1.5)
with raises(TypeError, match="neg expects a number, got str"):
    ev("neg", "a")

<a id="DOM-08"></a>
## DOM-08 · The bitwise operations take ints, as two's complement of infinite width, and shifts an int count

In [ ]:
assert ev("bitand", 12, 10) == 8 and ev("bitor", 12, 10) == 14 and ev("bitxor", 12, 10) == 6 and ev("bitnot", 5) == -6
assert ev("bitand", -1, 15) == 15 and ev("bitnot", -1) == 0 and ev("bitor", -16, 15) == -1
assert ev("shl", 3, 4) == 48 and ev("shr", -7, 1) == -4 and ev("shl", 1, 100) == 2 ** 100  # shr toward negative infinity
with raises(TypeError, match="bitand expects integers of one domain, got int and bytes"):
    ev("bitand", 1, b"\x01")
with raises(TypeError, match="bitnot expects an integer, got bool"):
    ev("bitnot", True)
with raises(ValueError, match="shl needs a non-negative count, got -1"):
    ev("shl", 1, -1)
with raises(TypeError, match="shr expects an integer count, got float"):
    ev("shr", 1, 1.0)
with raises(TypeError, match="shl expects an integer, got str"):
    ev("shl", "a", 1)
x = E.variable("x")
assert ev("bitand", x, 1, scope={"x": None}) is None and ev("shl", x, 1, scope={"x": None}) is None
assert ev("shl", 1, x, scope={"x": None}) is None

<a id="DOM-09"></a>
## DOM-09 · Signatures take the natives' domains and collections, so inference follows the domains an environment gives

In [ ]:
def infer(spec, environment=None):
    return E.DIALECT.infer(term(spec), environment)


y = E.variable("y")
assert infer(x.eq(y), {"x": D.Str, "y": D.Str}) is D.Bool and infer(x.lt(y), {"x": D.Int, "y": D.Int}) is D.Bool
with raises(TypeError, match="lt cannot take (bool, bool); it takes (T, T) -> bool for T in int | float | str | bytes"):
    infer(x.lt(y), {"x": D.Bool, "y": D.Bool})
assert infer(x.mul(y), {"x": D.Float, "y": FD.Anything}) is D.Float  # an unknown takes the other's domain
with raises(TypeError, match="add cannot take (str, str); it takes (T, T) -> T for T in int | float"):
    infer(x.add(y), {"x": D.Str, "y": D.Str})
with raises(TypeError, match="neg cannot take (str); it takes (T) -> T for T in int | float"):
    infer(E.operation("neg", "a"))
# The bitwise operations and shifts take ints.
assert infer(x.bitxor(y), {"x": D.Int, "y": D.Int}) is D.Int and infer(E.operation("bitnot", 1)) is D.Int
assert infer(E.operation("shr", x, 1), {"x": FD.Anything}) is D.Int
assert infer(E.operation("shl", 1, x), {"x": FD.OfUnion(D.Int, D.Float)}) is D.Int
with raises(TypeError, match="shl cannot take (int, float); it takes (int, int) -> int"):
    infer(E.operation("shl", 1, 1.5))
with raises(TypeError, match="shr cannot take (float, int)"):
    infer(E.operation("shr", 1.5, 1))
assert E.DIALECT.validate(term(E.operation("bitand", 1, 2)), core=True) == []
# A collection's operations take its items' domain: sum numbers, min and max ordered values.
assert infer(E.operation("sum", E.list_(1, 2))) is D.Int and infer(E.operation("max", E.list_("a", "b"))) is D.Str
with raises(TypeError, match="sum cannot take (list(str)); it takes (collection of number) -> number"):
    infer(E.operation("sum", E.list_("a", "b")))
with raises(TypeError, match="min cannot take (list(bool)); it takes (collection of ordered value) -> ordered value"):
    infer(E.operation("min", E.list_(True)))

<a id="DOM-10"></a>
## DOM-10 · An IEEE 754 value is a float up to binary64 and canonical text otherwise, which the format must hold

In [ ]:
holds = [  # a format, the values it holds, and values it does not
    ("binary16", [1.5, 65504.0, 2.0 ** -24, -0.0, math.inf, math.nan], [0.1, 1e10, 2.0 ** -25, "1.5"]),
    ("binary32", [0.10000000149011612], [0.1]),
    ("binary64", [0.1, 5e-324], ["0.1"]),
    ("binary128", ["0.1", "1E+2", "-0", "Infinity", "-Infinity", "NaN", "1.189731495357231765085759326628007E+4932"],
     ["0.10", "100", "1e2", "-NaN", "1E+5000", "0.1 "]),
    ("decimal64", ["1.50", "1.000000000000000E+16", "0E-398", "-0", "0.000001", "1E-7", "NaN"],
     ["1.5E+0", "12345678901234567", "1E+385", "1E-399", "0.0000001", "+1", "abc"]),
    ("decimal128", ["1.000000000000000000000000000000000", "2E+6111"], ["1E+6112"]),
    ("decimal32", ["1.000000E+7", "9.999999E+96", "1E-101"], ["12345678", "1E+97", "1E-102"]),
    # a format IEEE 754 does not name, by its radix, precision and emax: x87's extended format, and bfloat16
    ((2, 64, 16383), ["1", "0.1", "1.0000000000000000001"], ["1.00000000000000000001", "0.10"]),
    ((2, 8, 127), ["1", "1.016", "1.01"], ["1.015625", "1.0078125"]),
]
for format, inside, outside in holds:
    assert all(Ieee754.contains(format, v) for v in inside) and not any(Ieee754.contains(format, v) for v in outside), format
# A decimal value's text is General Decimal Arithmetic's scientific form, keeping the coefficient and the exponent.
data = [(False, 15, -1), (True, 150, -2), (False, 1, 2), (False, 123, -9), (False, 0, 3), (True, 0, 0)]
assert [Ieee754.text(Ieee754.Datum(None, *d)) for d in data] == ["1.5", "-1.50", "1E+2", "1.23E-7", "0E+3", "-0"]
assert Ieee754.text(Ieee754.NAN) == "NaN" and Ieee754.FORMATS["decimal64"] == (10, 16, 384)
assert Ieee754.FORMATS["decimal32"] == (10, 7, 96)  # IEEE 754-2019's, for C's _Decimal32
assert Ieee754.parameters("binary64") == (2, 53, 1023) and Ieee754.parameters((2, 8, 127)) == (2, 8, 127)
# A binary format's interchange encoding, big-endian, as an unsigned int: the canonical quiet NaN, subnormals, binary128.
assert Ieee754.to_bits("binary64", 1.5) == 4609434218613702656 and Ieee754.from_bits("binary64", 4609434218613702656) == 1.5
assert Ieee754.to_bits("binary64", -0.0) == 1 << 63 and Ieee754.to_bits("binary32", math.nan) == 0x7FC00000
assert Ieee754.from_bits("binary32", 0x7F800000) == math.inf and math.isnan(Ieee754.from_bits("binary32", 0xFF800001))
assert Ieee754.from_bits("binary16", 1) == 2.0 ** -24 and Ieee754.to_bits("binary16", 65504.0) == 0x7BFF
assert Ieee754.to_bits("binary16", 2.0 ** -24) == 1 and Ieee754.to_bits("binary16", 0.0) == 0
assert Ieee754.to_bits("binary16", -math.inf) == 0xFC00 and Ieee754.WIDTHS["binary16"] == 16
assert Ieee754.to_bits("binary128", "1") == int("3fff" + "00" * 14, 16)
assert Ieee754.from_bits("binary128", int("3fff8" + "0" * 27, 16)) == "1.5"

<a id="DOM-11"></a>
## DOM-11 · IEEE 754 arithmetic is exact, then rounded to the format by the rounding direction given

In [ ]:
ROUNDINGS = ("roundTiesToEven", "roundTiesToAway", "roundTowardPositive", "roundTowardNegative", "roundTowardZero")
EVEN, AWAY, UP, DOWN, ZERO = ROUNDINGS
inf, nan = math.inf, math.nan
X87, BF16 = (2, 64, 16383), (2, 8, 127)  # formats IEEE 754 does not name, by their parameters: values are text
TABLE = [  # format, rounding, operation, operands, result
    ("binary16", EVEN, "add", 1.0, 0.00048828125, 1.0),
    ("binary16", EVEN, "add", 1.0009765625, 0.00048828125, 1.001953125),
    ("binary16", AWAY, "add", 1.0, 0.00048828125, 1.0009765625),
    ("binary16", UP, "add", 1.0, 5.960464477539063e-08, 1.0009765625),
    ("binary16", DOWN, "add", 1.0, 5.960464477539063e-08, 1.0),
    ("binary16", ZERO, "add", 1.0, 5.960464477539063e-08, 1.0),
    ("binary16", DOWN, "add", -1.0, -5.960464477539063e-08, -1.0009765625),
    ("binary16", UP, "add", -1.0, -5.960464477539063e-08, -1.0),
    ("binary16", EVEN, "add", 65504.0, 65504.0, inf),
    ("binary16", ZERO, "add", 65504.0, 65504.0, 65504.0),
    ("binary16", UP, "add", 65504.0, 65504.0, inf),
    ("binary16", DOWN, "add", 65504.0, 65504.0, 65504.0),
    ("binary16", DOWN, "add", -65504.0, -65504.0, -inf),
    ("binary16", UP, "add", -65504.0, -65504.0, -65504.0),
    ("binary16", AWAY, "mul", 65504.0, 2.0, inf),
    ("binary16", EVEN, "mul", 6.103515625e-05, 0.0009765625, 5.960464477539063e-08),
    ("binary16", EVEN, "mul", 5.960464477539063e-08, 0.5, 0.0),
    ("binary16", AWAY, "mul", 5.960464477539063e-08, 0.5, 5.960464477539063e-08),
    ("binary16", EVEN, "add", 2047.0, 0.5, 2048.0),
    ("binary16", EVEN, "sub", 1.0, 1.0, 0.0),
    ("binary16", DOWN, "sub", 1.0, 1.0, -0.0),
    ("binary16", EVEN, "add", -0.0, -0.0, -0.0),
    ("binary16", EVEN, "add", 0.0, -0.0, 0.0),
    ("binary16", EVEN, "mul", -0.0, 5.0, -0.0),
    ("binary16", EVEN, "sub", inf, inf, nan),
    ("binary16", EVEN, "add", inf, 1.0, inf),
    ("binary16", EVEN, "add", 1.0, -inf, -inf),
    ("binary16", EVEN, "add", nan, 1.0, nan),
    ("binary16", EVEN, "mul", inf, 0.0, nan),
    ("binary16", EVEN, "mul", 0.0, -inf, nan),
    ("binary16", EVEN, "mul", inf, -2.0, -inf),
    ("binary16", EVEN, "mul", nan, 1.0, nan),
    ("binary16", EVEN, "neg", 1.5, None, -1.5),
    ("binary16", EVEN, "neg", nan, None, nan),
    ("binary16", EVEN, "neg", -0.0, None, 0.0),
    ("binary32", EVEN, "add", 16777216.0, 1.0, 16777216.0),
    ("binary32", UP, "add", 16777216.0, 1.0, 16777218.0),
    ("binary32", EVEN, "mul", 0.10000000149011612, 3.0, 0.30000001192092896),
    ("binary64", ZERO, "add", 0.1, 0.2, 0.3),
    ("binary64", UP, "add", 0.1, 0.2, 0.30000000000000004),
    ("binary64", DOWN, "mul", 0.3333333333333333, 3.0, 0.9999999999999999),
    ("binary128", EVEN, "add", "0.1", "0.2", "0.30000000000000000000000000000000004"),
    ("binary128", EVEN, "mul", "1E+4932", "1E+1", "Infinity"),
    ("binary128", ZERO, "mul", "1E+4932", "1E+1", "1.189731495357231765085759326628007E+4932"),
    ("binary128", EVEN, "sub", "1", "1", "0"),
    ("binary128", DOWN, "sub", "1", "1", "-0"),
    ("binary128", EVEN, "neg", "-0", None, "0"),
    ("binary128", EVEN, "add", "Infinity", "1", "Infinity"),
    ("binary128", EVEN, "mul", "1E-4960", "1E-10", "0"),
    ("decimal64", EVEN, "add", "1.5", "1.50", "3.00"),
    ("decimal64", EVEN, "mul", "1.5", "1.50", "2.250"),
    ("decimal64", EVEN, "add", "9999999999999999", "1", "1.000000000000000E+16"),
    ("decimal64", EVEN, "add", "9999999999999999", "0.5", "1.000000000000000E+16"),
    ("decimal64", AWAY, "add", "1000000000000000", "0.5", "1000000000000001"),
    ("decimal64", EVEN, "add", "1000000000000000", "0.5", "1000000000000000"),
    ("decimal64", UP, "add", "1000000000000000", "0.1", "1000000000000001"),
    ("decimal64", DOWN, "add", "-1000000000000000", "-0.1", "-1000000000000001"),
    ("decimal64", ZERO, "add", "-1000000000000000", "-0.9", "-1000000000000000"),
    ("decimal64", EVEN, "mul", "1E+369", "1E+10", "1.0000000000E+379"),
    ("decimal64", EVEN, "add", "9.999999999999999E+384", "1E+369", "Infinity"),
    ("decimal64", ZERO, "add", "9.999999999999999E+384", "1E+369", "9.999999999999999E+384"),
    ("decimal64", EVEN, "mul", "1E-398", "0.1", "0E-398"),
    ("decimal64", UP, "mul", "1E-398", "0.1", "1E-398"),
    ("decimal64", EVEN, "mul", "1E-390", "0.00001", "1E-395"),
    ("decimal64", EVEN, "sub", "1.5", "1.5", "0.0"),
    ("decimal64", DOWN, "sub", "1.5", "1.5", "-0.0"),
    ("decimal64", EVEN, "add", "-0", "-0", "-0"),
    ("decimal64", EVEN, "mul", "-0.0", "1E+5", "-0E+4"),
    ("decimal64", EVEN, "mul", "1E+369", "0E+10", "0E+369"),
    ("decimal64", EVEN, "add", "NaN", "1", "NaN"),
    ("decimal64", EVEN, "mul", "Infinity", "0", "NaN"),
    ("decimal64", EVEN, "neg", "1.50", None, "-1.50"),
    ("decimal64", EVEN, "add", "0.000001", "1E-7", "0.0000011"),
    ("decimal64", EVEN, "mul", "123", "1E+5", "1.23E+7"),
    ("decimal128", EVEN, "mul", "0.1", "3", "0.3"),
    ("decimal128", EVEN, "add", "1E+6111", "1E+6111", "2E+6111"),
    ("decimal128", EVEN, "add", "1", "1E-40", "1.000000000000000000000000000000000"),
    ("decimal32", EVEN, "add", "9999999", "1", "1.000000E+7"),
    ("decimal32", EVEN, "mul", "1.5", "1.50", "2.250"),
    ("decimal32", EVEN, "add", "9.999999E+96", "1E+90", "Infinity"),
    (X87, EVEN, "add", "1", "1E-19", "1.0000000000000000001"),  # rounded to 64 bits of precision, not binary128's 113
    (X87, EVEN, "mul", "1E+4932", "10", "Infinity"),
    (BF16, EVEN, "add", "1", "0.00390625", "1"),  # a tie, to even
    (BF16, EVEN, "add", "1", "0.01171875", "1.016"),  # a tie, to even: 1.015625, the shortest text that rounds back to it
    (BF16, EVEN, "mul", "3E+38", "2", "Infinity"),
]
CONTEXTS = {"decimal32": (7, 96), "decimal64": (16, 384), "decimal128": (34, 6144)}
MODES = dict(zip(ROUNDINGS, [decimal.ROUND_HALF_EVEN, decimal.ROUND_HALF_UP, decimal.ROUND_CEILING, decimal.ROUND_FLOOR,
                               decimal.ROUND_DOWN]))


def alike(x, y):
    """The same value: floats by sign too, NaNs alike."""
    if isinstance(x, float) and isinstance(y, float):
        return (math.isnan(x) and math.isnan(y)) or (x == y and math.copysign(1, x) == math.copysign(1, y))
    return x == y


for format, rounding, operation, a, b, expected in TABLE:
    result = Ieee754.operate(operation, format, rounding, [v for v in (a, b) if v is not None])
    assert alike(result, expected), (format, a, b, result)
    # Python's decimal module and the host's floats agree.
    if format in CONTEXTS:
        precision, emax = CONTEXTS[format]
        context = decimal.Context(prec=precision, Emax=emax, Emin=1 - emax, clamp=1, rounding=MODES[rounding], traps=[])
        left, right = decimal.Decimal(a), decimal.Decimal(b or 0)
        negate = lambda value, _: value.copy_negate()  # noqa: E731
        reference = {"add": context.add, "sub": context.subtract, "mul": context.multiply}.get(operation, negate)(left, right)
        assert ("NaN" if reference.is_nan() else str(reference)) == expected, (format, a, b, reference)
    elif rounding == EVEN and format in ("binary16", "binary32") and operation != "neg":
        exact = {"add": a + b, "sub": a - b, "mul": a * b}[operation]
        code = "e" if format == "binary16" else "f"
        try:
            reference = struct.unpack(code, struct.pack(code, exact))[0]
        except OverflowError:
            reference = math.copysign(math.inf, exact)
        assert alike(reference, expected), (format, a, b, reference)
# Division, which Patterns has no operation for and C has (see C), by the same engine; decimals as divide does.
DIVISIONS = [  # format, dividend, divisor, quotient, rounding ties to even
    ("binary64", nan, 1.0, nan), ("binary64", inf, inf, nan), ("binary64", inf, -2.0, -inf), ("binary64", 1.0, inf, 0.0),
    ("binary64", -1.0, inf, -0.0), ("binary64", 1.0, 0.0, inf), ("binary64", -1.0, 0.0, -inf), ("binary64", 0.0, 0.0, nan),
    ("binary64", 1.0, 3.0, 0.3333333333333333), ("binary32", 1.0, 3.0, 0.3333333432674408), ("decimal64", "1", "4", "0.25"),
    ("decimal64", "100", "10", "10"), ("decimal64", "1E+2", "1", "1E+2"), ("decimal64", "1", "3", "0.3333333333333333"),
    ("decimal64", "0", "5", "0"), ("decimal64", "0.00", "5", "0.00"), ("decimal64", "1", "Infinity", "0E-398"),
    ("decimal64", "Infinity", "2", "Infinity"), ("decimal64", "1", "0", "Infinity"), ("decimal64", "0", "0", "NaN"),
    ("decimal64", "-1", "8", "-0.125"), ("decimal64", "2.0", "4", "0.5"),
    ("decimal128", "1", "7", "0.1428571428571428571428571428571429"), (X87, "1", "3", "0.33333333333333333334"),
]
for format, a, b, expected in DIVISIONS:
    assert alike(Ieee754.operate("div", format, EVEN, [a, b]), expected), (format, a, b)
    if format in CONTEXTS:
        precision, emax = CONTEXTS[format]
        context = decimal.Context(prec=precision, Emax=emax, Emin=1 - emax, clamp=1, traps=[])
        reference = context.divide(decimal.Decimal(a), decimal.Decimal(b))
        assert ("NaN" if reference.is_nan() else str(reference)) == expected, (format, a, b, reference)
# Evaluation's floats keep the host's arithmetic, which binary64 rounding ties to even is.
assert ev("add", 0.1, 0.2) == 0.30000000000000004 == Ieee754.operate("add", "binary64", EVEN, [0.1, 0.2])
# Conversions between formats and from and to integers, by the same engine (C's and SystemVerilog's casts use them).
assert Ieee754.convert("binary64", 0.1, "binary32", EVEN) == 0.10000000149011612
assert Ieee754.convert("binary64", 0.1, "decimal64", EVEN) == "0.1000000000000000"
assert Ieee754.convert("binary64", 100.0, "decimal64", EVEN) == "100" and Ieee754.convert("binary64", 0.5, "decimal64", EVEN) == "0.5"
assert Ieee754.convert("decimal64", "1.50", "decimal128", EVEN) == "1.50" and Ieee754.convert("decimal64", "0.1", "binary64", EVEN) == 0.1
assert Ieee754.convert("binary64", math.nan, "binary128", EVEN) == "NaN" and Ieee754.convert("binary64", -math.inf, "decimal64", EVEN) == "-Infinity"
assert Ieee754.from_integer(2 ** 24 + 1, "binary32", EVEN) == 16777216.0 and Ieee754.from_integer(2 ** 24 + 1, "binary32", UP) == 16777218.0
assert Ieee754.from_integer(10 ** 20, "decimal64", EVEN) == "1.000000000000000E+20" and Ieee754.from_integer(-7, "decimal64", EVEN) == "-7"
assert Ieee754.to_integer("binary64", 2.5, EVEN) == 2 and Ieee754.to_integer("binary64", -2.7, ZERO) == -2
assert Ieee754.to_integer("decimal64", "2.5", EVEN) == 2 and Ieee754.to_integer("binary64", math.inf, EVEN) == math.inf
assert Ieee754.to_integer("binary64", -math.inf, EVEN) == -math.inf
assert Ieee754.convert(X87, Ieee754.convert("binary64", 0.1, X87, EVEN), "binary64", EVEN) == 0.1  # x87 holds binary64's
assert Ieee754.from_integer(2 ** 64 + 1, X87, EVEN) == "18446744073709551616" and Ieee754.to_integer(X87, "2.5", EVEN) == 2
with raises(ValueError, match="NaN has no integer value"):
    Ieee754.to_integer("binary64", math.nan, EVEN)

<a id="DOM-12"></a>
## DOM-12 · IEEE 754 values compare by totalOrder, cohorts by exponent, and NaNs equal each other

In [ ]:
COMPARISONS = [  # a format, two of its values, and how they compare: -1, 0 or 1, or None when unordered
    ("decimal64", "1.5", "1.5", 0), ("decimal64", "1.5", "1.50", 1), ("decimal64", "1.50", "1.5", -1),  # one value, two cohorts
    ("decimal64", "-1.5", "-1.50", -1), ("decimal64", "1.5", "2", -1), ("decimal64", "-1", "-2", 1), ("decimal64", "-0", "0", -1),
    ("decimal64", "-Infinity", "-9.999999999999999E+384", -1), ("decimal64", "9.999999999999999E+384", "Infinity", -1),
    ("decimal64", "Infinity", "Infinity", 0), ("decimal64", "NaN", "NaN", 0), ("decimal64", "NaN", "1", None),
    ("decimal64", "1", "NaN", None), ("decimal64", "1", "-1", 1), ("binary128", "0.1", "0.2", -1), ("binary128", "0.1", "0.1", 0),
    ("binary16", -0.0, 0.0, -1), ("binary16", math.nan, math.nan, 0), ("decimal32", "1.5", "1.50", 1),
    ((2, 64, 16383), "0.1", "0.2", -1),
]
assert [Ieee754.compare(format, a, b) for format, a, b, _ in COMPARISONS] == [expected for *_, expected in COMPARISONS]
# A binary128 value's text is the shortest that rounds back to it.
EVEN = "roundTiesToEven"
assert Ieee754.operate("mul", "binary128", EVEN, ["3", "0.5"]) == "1.5"
assert Ieee754.operate("add", "binary128", EVEN, ["0.1", "0.2"]) == "0.30000000000000000000000000000000004"
assert Ieee754.operate("mul", "binary128", EVEN, ["0.5", "0.5"]) == "0.25"
assert Ieee754.operate("neg", "binary128", EVEN, ["0.49"]) == "-0.49"
assert Ieee754.operate("mul", "binary128", EVEN, ["1E+48", "1E+1"]) == "1E+49"  # the nearest is below 1E+49

<a id="DOM-17"></a>
## DOM-17 · The standard dialects own the kinds and domains, with their inferred properties, read through the framework's interface

In [ ]:
# Each standard dialect: a singleton of its own schema, a branch of the framework's kind union named by its tag, with
# no terms; the edition of its standard it covers, by designation, its formal reference's parts held apart; its union
# of kinds the branch of the framework's named by its tag.
KINDS = {ISOIEC11404: ("isoiec11404", "ISO/IEC 11404:2007", [
             "boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"]),
         IEEE754: ("ieee754", "IEEE Std 754-2019", ["ieee754"]), IEEE1164: ("ieee1164", "IEEE Std 1164-1993", ["ieee1164"]),
         Unicode: ("unicode", "The Unicode Standard", ["unicode"])}
EDITIONS = {  # each standard's formal reference: its bodies by reverse-domain name, its number and its year
    "ISO/IEC 11404:2007": {"bodies": ("org.iso", "ch.iec"), "number": "11404", "revision": ("year", 2007)},
    "IEEE Std 754-2019": {"bodies": ("org.ieee",), "number": "754", "revision": ("year", 2019)},
    "IEEE Std 1164-1993": {"bodies": ("org.ieee",), "number": "1164", "revision": ("year", 1993)},
    "The Unicode Standard": {"bodies": ("org.unicode",)}}  # unversioned, as the dialect reads it
Dialects = store.registered("Expressions.Framework.Dialect")
for module, (tag, standard, kinds) in KINDS.items():
    path = f"Expressions.{DIALECTS[module]}.Expressions"
    covered = FS.standards(singletons[DIALECTS[module]])  # its edition, its kinds none: it has no terms
    assert list(covered) == [standard] and FS.properties(covered[standard]) == EDITIONS[standard]
    assert FS.kinds(covered[standard]) == [] and defined[DIALECTS[module]]["standard"] is covered[standard]
    assert module.__name__ == f"mbse.{path}"  # its package, by its path,
    assert module.Expressions.install is module.install  # exporting its module's
    union = {b.name: b.schema for b in DomainKind.branches}[tag]
    assert union is store.registered(f"{path}.DomainKind") and [b.name for b in union.branches] == kinds
    dialect = {b.name: b.schema for b in Dialects.branches}[tag]
    assert dialect is store.registered(f"{path}.Dialect") and dialect.ref and dialect.singleton == path
    assert list(dialect.adjacencies) == ["domains", "operations", "standards"]
    assert FD.operations(singletons[DIALECTS[module]]) == {}
    with raises(LookupError, match=f"no schema registered as '{path}.Term'"):
        store.registered(f"{path}.Term")
# The kinds: their parameters, then the properties inferred from them, all optional.
assert list(store.registered("Expressions.ISOIEC11404.Expressions.OfInteger").properties) == [
    "width", "signed", "overflow", "least", "greatest", "ordered"]
assert list(store.registered("Expressions.IEEE754.Expressions.OfIeee754").properties) == [
    "format", "rounding", "width", "radix", "precision", "emax", "ordered"]
assert list(store.registered("Expressions.ISOIEC11404.Expressions.OfModulo").properties) == [
    "modulus", "least", "greatest", "ordered"]
assert [list(store.registered(name).properties) for name in [
    "Expressions.ISOIEC11404.Expressions.OfOctet",
    "Expressions.ISOIEC11404.Expressions.OfBoolean", "Expressions.ISOIEC11404.Expressions.OfBitstring",
    "Expressions.ISOIEC11404.Expressions.OfOctetstring", "Expressions.ISOIEC11404.Expressions.OfState",
    "Expressions.ISOIEC11404.Expressions.OfPacked", "Expressions.IEEE1164.Expressions.OfIeee1164",
    "Expressions.Unicode.Expressions.OfUnicode"]] == [
    ["ordered"], ["ordered"], ["width", "ordered"], ["width", "ordered"], ["members", "ordered"], ["codes", "ordered"],
    ["width", "ordered"], ["ordered"]]
# Each domain's kind, its dialect's tag then its own, through its dialect's union, and its properties, parameters and
# inferred ones alike, by name.
assert {name: FD.kind_of(domain)[:2] for name, domain in domains.items()} == {
    "boolean": ("isoiec11404", "boolean"), "integer": ("isoiec11404", "integer"),
    "octetstring": ("isoiec11404", "octetstring"), "octet": ("isoiec11404", "octet"),
    "modulo(256)": ("isoiec11404", "modulo"), "binary64": ("ieee754", "ieee754"),
    "binary32": ("ieee754", "ieee754"), "std_logic": ("ieee1164", "ieee1164"), "string": ("unicode", "unicode")}
assert FD.kind_of(domains["octet"])[2].as_schema() is store.registered("Expressions.ISOIEC11404.Expressions.OfOctet")
assert FD.kind_of(domains["modulo(256)"])[2].as_schema() is store.registered("Expressions.ISOIEC11404.Expressions.OfModulo")
PROPERTIES = {
    "boolean": {"ordered": False}, "integer": {"signed": True, "overflow": "raise", "ordered": True},  # unbounded
    "octetstring": {"ordered": True},
    "octet": {"ordered": True}, "modulo(256)": {"modulus": 256, "least": 0, "greatest": 255, "ordered": True},
    "binary64": {"format": "binary64", "rounding": "roundTiesToEven", "width": 64, "radix": 2, "precision": 53, "emax": 1023,
                 "ordered": True},
    "binary32": {"format": "binary32", "rounding": "roundTiesToEven", "width": 32, "radix": 2, "precision": 24, "emax": 127,
                 "ordered": True},
    "std_logic": {"ordered": False}, "string": {"ordered": True}}
assert {name: FD.properties(domain) for name, domain in domains.items()} == PROPERTIES
assert FD.properties(state) == {"members": ("IDLE", "RUN"), "ordered": False}  # a container as a tuple
assert FD.properties(coded) == {"codes": (0, 1), "ordered": True}
# The inferred properties, populated by hand: IEEE 754's agree with its formats' parameters.
assert all((p["radix"], p["precision"], p["emax"]) == Ieee754.FORMATS[p["format"]] and p["width"] == Ieee754.WIDTHS[p["format"]]
           for p in (PROPERTIES["binary64"], PROPERTIES["binary32"]))
# A domain's name, its dialect's `Defines` entry's; None until a dialect defines it.
assert [FD.name_of(domain) for domain in domains.values()] == list(domains) and FD.name_of(coded) == "coded"
with store.transaction():
    loose = store.Expressions.Framework.Domain(lambda d: d.kind(lambda k: k.as_value(lambda a: a.as_union(
        lambda u: u.unicode(lambda v: v.unicode(lambda e: e))))))
    assert FD.name_of(loose) is None
    store.Expressions.Unicode.Expressions.Dialect(singletons["Unicode"], lambda b: b.domains({"loose": lambda e: e.domain(loose)}))
assert FD.name_of(loose) == "loose" and FD.properties(loose) == {}  # no property set
with raises(ValueError, match="a domain needs a kind"):
    FD.kind_of(empty)
with raises(ValueError, match="a domain needs a kind"):
    FD.properties(empty)
# A dialect's operations by name, in order, each its expression schema, and each one's arity: Patterns', which its
# signatures for inference name in the same order, each of the same arity.
patterns = store.transaction(Patterns.define)[0]["dialect"]  # again: nothing
operations = FD.operations(patterns)
assert list(operations) == list(D.SIGNATURES) and operations["add"] is store.registered("Expressions.Patterns.Expressions.add")
assert {name: FD.arity(operation) for name, operation in operations.items()} == {
    name: signature.arity() for name, signature in D.SIGNATURES.items()}
# Patterns owns no domain, and holds no kind and no domain's schema; its family is its own, not the framework's: its
# schemas, singleton and package are at `Expressions.Patterns.Expressions`, and nothing at the framework's old path.
assert list(patterns.domains) == [] and list(DomainKind.branches[0].schema.branches) and "patterns" not in [b.name for b in DomainKind.branches]
assert Patterns.__name__ == "mbse.Expressions.Patterns.Expressions" and patterns is store.singleton("Expressions.Patterns.Expressions")
for name in ["Expressions.Patterns.Expressions.octet", "Expressions.Patterns.Expressions.OfInteger", "Expressions.Patterns.Expressions.Packs",
             "Expressions.Framework.Patterns.Term", "Expressions.Framework.Patterns.Dialect"]:
    with raises(LookupError, match=f"no schema registered as '{name}'"):
        store.registered(name)

<a id="DOM-18"></a>
## DOM-18 · A language's own kinds are its dialect's: SystemVerilog's logic and bit, Python3's, TypeScript5's, MATLAB's and Excel's strings, the tokens and Excel's errors

In [ ]:
# Each dialect with kinds of its own adds its union of them to the framework's, as the branch named by its tag: a
# language's own types (Decided, B and H), each kind's parameters, then its inferred properties, all optional.
languages = Builders.Store()
for install in [*INSTALLS, Python3.install, Matlab.install, Excel.install, Latex.install, C.install, Cpp.install,
                SystemVerilog.install, TypeScript5.install, PythonTypes.install, PythonPatterns.install,  # each new
                TypeScriptTypes.install, TypeScriptPatterns.install, CTypes.install, CppTypes.install,  # dialect after
                CPreprocessor.install, PythonRegExp.install, TypeScriptRegExp.install, CppRegExp.install,  # its family's,
                PosixBRE.install, PosixERE.install]:  # POSIX's after C++'s; kinds of domain none
    languages.transaction(install)
LANGUAGES = {  # each dialect's tag, its path, and its kinds' properties, by tag
    "python3": ("Expressions.Python.Python3", {"str": ("OfStr", ["ordered"]), "nonetype": ("OfNoneType", ["ordered"]),
                                                "ellipsis": ("OfEllipsis", ["ordered"])}),
    "matlab": ("Expressions.MATLAB.Expressions", {"string": ("OfString", ["ordered"])}),
    "excel": ("Expressions.Excel.Expressions", {"text": ("OfText", ["ordered"]), "error": ("OfError", ["members", "ordered"])}),
    "c": ("Expressions.C.Expressions", {"nullptr_t": ("OfNullptr", ["ordered"])}),
    "cpp": ("Expressions.Cpp.Expressions", {"nullptr_t": ("OfNullptr", ["ordered"])}),
    "systemverilog": ("Expressions.SystemVerilog.Expressions", {
        "logic": ("OfLogic", ["width", "signed", "least", "greatest", "ordered"]),
        "bit": ("OfBit", ["width", "signed", "least", "greatest", "ordered"])}),
    "typescript5": ("Expressions.TypeScript.TypeScript5", {"string": ("OfString", ["ordered"]), "null": ("OfNull", ["ordered"])}),
}
Kinds = languages.registered("Expressions.Framework.DomainKind")
assert [b.name for b in Kinds.branches] == ["isoiec11404", "ieee754", "ieee1164", "unicode", *LANGUAGES]  # LaTeX: none
for tag, (path, kinds) in LANGUAGES.items():
    union = {b.name: b.schema for b in Kinds.branches}[tag]
    assert union is languages.registered(f"{path}.DomainKind")
    assert {b.name: (b.schema.name, list(b.schema.properties)) for b in union.branches} == {
        kind: (f"{path}.{schema}", names) for kind, (schema, names) in kinds.items()}
members = languages.registered("Expressions.Excel.Expressions.OfError").properties["members"].schema
assert (members.index, members.element) == (languages.registered("Schemas.OfInt"), languages.registered("Schemas.OfStr"))
# C's nullptr_t and C++'s are each their own dialect's (API, 35).
assert languages.registered("Expressions.C.Expressions.OfNullptr") is not languages.registered("Expressions.Cpp.Expressions.OfNullptr")
# A domain of each, written by hand in a transaction and owned by its dialect by name, valid against its schema;
# `ordered` true of the strings and SystemVerilog's logic, false of the tokens and Excel's errors.
ERRORS = ["#DIV/0!", "#N/A", "#NAME?", "#NULL!", "#NUM!", "#REF!", "#VALUE!"]
WRITTEN = {  # by its dialect's tag and its name: its kind, its properties written, and as read
    ("systemverilog", "logic8"): ("logic", lambda e: e.width(8).signed(False).least(0).greatest(255).ordered(True),
                                  {"width": 8, "signed": False, "least": 0, "greatest": 255, "ordered": True}),
    ("python3", "str"): ("str", lambda e: e.ordered(True), {"ordered": True}),
    ("python3", "NoneType"): ("nonetype", lambda e: e.ordered(False), {"ordered": False}),
    ("python3", "ellipsis"): ("ellipsis", lambda e: e.ordered(False), {"ordered": False}),
    ("typescript5", "string"): ("string", lambda e: e.ordered(True), {"ordered": True}),
    ("typescript5", "null"): ("null", lambda e: e.ordered(False), {"ordered": False}),
    ("matlab", "string"): ("string", lambda e: e.ordered(True), {"ordered": True}),
    ("excel", "text"): ("text", lambda e: e.ordered(True), {"ordered": True}),
    ("c", "nullptr_t"): ("nullptr_t", lambda e: e.ordered(False), {"ordered": False}),
    ("cpp", "nullptr_t"): ("nullptr_t", lambda e: e.ordered(False), {"ordered": False}),
    ("excel", "error"): ("error", lambda e: e.members(ERRORS).ordered(False), {"members": tuple(ERRORS), "ordered": False}),
}


def schema_at(path):
    """The builder of a schema registered under a dotted path, by its parts."""
    found = languages
    for part in path.split("."):
        found = getattr(found, part)
    return found


LanguageDomain = languages.registered("Expressions.Framework.Domain")
with languages.transaction():
    written = {key: languages.Expressions.Framework.Domain(lambda d, tag=key[0], kind=kind, spec=spec: d.kind(
        lambda k: k.as_value(lambda a: a.as_union(lambda u: getattr(u, tag)(lambda v: getattr(v, kind)(spec))))))
        for key, (kind, spec, _) in WRITTEN.items()}
    for (tag, name), domain in written.items():  # each owned: written with its owner
        path = LANGUAGES[tag][0]
        schema_at(f"{path}.Dialect")(languages.singleton(path), lambda b, name=name, domain=domain: b.domains({
            name: lambda e: e.domain(domain)}))
for (tag, name), domain in written.items():
    kind, _, read = WRITTEN[tag, name]
    assert Validators.Validate(languages)(LanguageDomain, domain) == [], (tag, name)
    dialect, found, record = FD.kind_of(domain)  # the dialect's tag first: C's nullptr_t is not C++'s
    assert (dialect, found) == (tag, kind) and record.as_schema() is languages.registered(f"{LANGUAGES[tag][0]}.{LANGUAGES[tag][1][kind][0]}")
    assert FD.properties(domain) == read and FD.name_of(domain) == name
assert [FD.kind_of(written[tag, "nullptr_t"])[:2] for tag in ("c", "cpp")] == [("c", "nullptr_t"), ("cpp", "nullptr_t")]

<a id="DOM-19"></a>
## DOM-19 · C's, C++'s and the ABI dialects' domains are stored: C's and C++'s defines write the types their standards fix, each ABI dialect's those they leave to the implementation, by name, of ISO/IEC 11404's and IEEE 754's kinds or a token's, with their properties and schemas; a holder names its ABI, and C's and C++'s types are read through it

In [ ]:
# C's, C++'s and the two ABI dialects' defines, each run after the installs in a transaction of its own: their domains,
# the types C's and C++'s standards fix (C's and C++'s own) and those the standards leave to the implementation, as each
# ABI fixes them (each ABI dialect's), owned by name, in order, each of ISO/IEC 11404's kinds, IEEE 754's or its own
# nullptr_t, with its parameters and its inferred properties; each but a token related to its schema, an OfAny of the
# scalar it narrows, both ways (Decided, Q).
machine = Builders.Store()
MACHINE = [*INSTALLS, C.install, Cpp.install, AMD64.install, AAPCS64.install,  # the ABIs after the standard dialects
           CTypes.install, CppTypes.install, CPreprocessor.install]  # C's and C++'s new dialects after their families'
for install in MACHINE:
    machine.transaction(install)
DEFINES = [C, Cpp, AMD64, AAPCS64]
frames = {module: machine.transaction(module.define)[0] for module in DEFINES}  # the defines after the installs
ISO, IEEE = "isoiec11404", "ieee754"


def signed(width):
    """A signed integer's kind and properties, and the scalar its schema narrows: 11404's integer, two's complement,
    raising on overflow, as C leaves it undefined."""
    return (ISO, "integer"), {"width": width, "signed": True, "overflow": "raise", "least": -2 ** (width - 1),
                              "greatest": 2 ** (width - 1) - 1, "ordered": True}, "int"


def unsigned(width):
    """An unsigned integer's: 11404's modulo, wrapping."""
    return (ISO, "modulo"), {"modulus": 2 ** width, "least": 0, "greatest": 2 ** width - 1, "ordered": True}, "int"


def ieee754(width, format=None, parameters=None):
    """An IEEE 754 domain's, of a format by its name or, where IEEE 754 names none, by its parameters: its values floats
    for binary16 to binary64, else text."""
    radix, precision, emax = Ieee754.FORMATS[format] if format else parameters
    named = {"format": format} if format else {}
    return (IEEE, IEEE), {**named, "rounding": "roundTiesToEven", "width": width, "radix": radix, "precision": precision,
                          "emax": emax, "ordered": True}, "float" if format in ("binary16", "binary32", "binary64") else "str"


BOOL = (ISO, "boolean"), {"ordered": False}, "bool"
EXACT = {**{f"int{w}_t": signed(w) for w in (8, 16, 32, 64)}, **{f"uint{w}_t": unsigned(w) for w in (8, 16, 32, 64)}}
C_DOMAINS = {"bool": BOOL, **EXACT, "_Float16": ieee754(16, "binary16"), "_Float32": ieee754(32, "binary32"),
             "_Float64": ieee754(64, "binary64"), "_Float128": ieee754(128, "binary128"),
             "_Decimal32": ieee754(32, "decimal32"), "_Decimal64": ieee754(64, "decimal64"),
             "_Decimal128": ieee754(128, "decimal128"), "nullptr_t": (("c", "nullptr_t"), {"ordered": False}, None)}
CPP_DOMAINS = {"bool": BOOL, **EXACT, "std::float16_t": ieee754(16, "binary16"), "std::float32_t": ieee754(32, "binary32"),
               "std::float64_t": ieee754(64, "binary64"), "std::float128_t": ieee754(128, "binary128"),
               "std::bfloat16_t": ieee754(16, parameters=(2, 8, 127)),  # C++23's, by IEEE 754's parameters, no format
               "nullptr_t": (("cpp", "nullptr_t"), {"ordered": False}, None)}  # C++'s own token, not C's


def abi(char, wchar, extended):
    """An ABI's domains, in the order C's standard lists the types, of its plain char, wchar_t and long double."""
    return {"char": char, "signed char": signed(8), "unsigned char": unsigned(8), "short": signed(16),
            "unsigned short": unsigned(16), "int": signed(32), "unsigned int": unsigned(32), "long": signed(64),
            "unsigned long": unsigned(64), "long long": signed(64), "unsigned long long": unsigned(64),
            "float": ieee754(32, "binary32"), "double": ieee754(64, "binary64"), "long double": extended, "wchar_t": wchar,
            "char8_t": unsigned(8), "char16_t": unsigned(16), "char32_t": unsigned(32), "size_t": unsigned(64),
            "ptrdiff_t": signed(64), "intptr_t": signed(64), "uintptr_t": unsigned(64),
            **{f"int_least{w}_t": signed(w) for w in (8, 16, 32, 64)},
            **{f"uint_least{w}_t": unsigned(w) for w in (8, 16, 32, 64)},
            "_Float32x": ieee754(64, "binary64"), "_Float64x": extended}


X87 = ieee754(80, parameters=(2, 64, 16383))  # x87's extended format: IEEE 754's binary64 extended, no format
DEFINED = {  # each dialect's path, and its domains by name, in order
    C: ("Expressions.C.Expressions", C_DOMAINS), Cpp: ("Expressions.Cpp.Expressions", CPP_DOMAINS),
    AMD64: ("Expressions.SystemV.AMD64", abi(signed(8), signed(32), X87)),  # char signed, wchar_t an int
    AAPCS64: ("Expressions.Arm.AAPCS64", abi(unsigned(8), unsigned(32), ieee754(128, "binary128"))),  # unsigned
}


def spelled(name):
    """A domain's schema's name, and its define's symbol for it: its type's, spelled as a dotted identifier, with no
    leading `_`, which mbse-schemas' frames do not take (`unsigned long` as `unsigned_long`, `_Float16` as `Float16`,
    `std::float16_t` as `std_float16_t`)."""
    return name.replace("::", "_").replace(" ", "_").lstrip("_")


MachineDomain = machine.registered("Expressions.Framework.Domain")
for module, (path, expected) in DEFINED.items():
    assert module.Expressions.define is module.define and module.Expressions.install is module.install  # exported
    dialect = machine.singleton(path)
    assert frames[module]["dialect"] is dialect and [str(e.name) for e in dialect.domains] == list(expected), path
    assert list(FD.domains(dialect)) == list(expected) and FD.operations(dialect) == {}  # no operations
    for name, (kind, properties, scalar) in expected.items():
        domain = dialect.domains({"name": name})
        assert frames[module][spelled(name)] is domain and FD.domains(dialect)[name] is domain
        assert Validators.Validate(machine)(MachineDomain, domain) == [], (path, name)
        assert FD.kind_of(domain)[:2] == kind and FD.properties(domain) == properties and FD.name_of(domain) == name, (path, name)
        if scalar is None:  # a token: no basic scalar holds it, so no schema
            assert list(domain.schema) == []
            with raises(LookupError, match=f"no schema registered as '{path}.{name}'"):
                machine.registered(f"{path}.{name}")
            continue
        schema = machine.registered(f"{path}.{spelled(name)}")
        assert schema.schemas == (machine.registered(f"Schemas.Of{scalar.capitalize()}"),) and schema.validate() == []
        assert [e.schema for e in domain.schema] == [schema]  # HasDomain, both ways
        read = [e.domain for e in Builders.OfAny.Builder(schema, False, machine).domain]
        assert [(FD.name_of(d), FD.kind_of(d)[:2], FD.properties(d)) for d in read] == [(name, kind, properties)]
# The inferred properties, populated by hand: IEEE 754's agree with its formats' parameters, an integer's bounds with its
# width, a modulo's with its modulus.
for _, expected in DEFINED.values():
    for kind, properties, _ in expected.values():
        if "format" in properties:
            assert (properties["radix"], properties["precision"], properties["emax"]) == Ieee754.FORMATS[properties["format"]]
        if "modulus" in properties:
            assert (properties["least"], properties["greatest"]) == (0, properties["modulus"] - 1)
# Each ABI dialect: a singleton of its own schema, a branch of the framework's kind union of dialects by its tag, with
# no terms and no kinds of its own; the standard it covers by its title, Arm's by its abbreviation too; a holder names it
# as its `abi`, which its schema declares,
# anonymous, and no other dialect does.
Dialects = machine.registered("Expressions.Framework.Dialect")
for module, tag, path, standard, properties in [
    (AMD64, "amd64", "Expressions.SystemV.AMD64", "System V Application Binary Interface, AMD64 Architecture Processor Supplement",
     {}),  # its editors only: no body
    (AAPCS64, "aapcs64", "Expressions.Arm.AAPCS64", "Procedure Call Standard for the Arm 64-bit Architecture (AArch64)",
     {"bodies": ("com.arm",), "names": ("AAPCS64",)}),
]:
    covered = FS.standards(machine.singleton(path))
    assert list(covered) == [standard] and FS.properties(covered[standard]) == properties
    assert FS.tags(covered[standard]) == (standard, *properties.get("names", ())) and frames[module]["standard"] is covered[standard]
    assert module.__name__ == f"mbse.{path}"
    dialect = {b.name: b.schema for b in Dialects.branches}[tag]
    assert dialect is machine.registered(f"{path}.Dialect") and dialect.ref and dialect.singleton == path
    assert list(dialect.adjacencies) == ["domains", "operations", "standards"] and tag not in [b.name for b in DomainKind.branches]
    for missing in ("Term", "DomainKind", "Expression"):
        with raises(LookupError, match=f"no schema registered as '{path}.{missing}'"):
            machine.registered(f"{path}.{missing}")
assert [b.name for b in Dialects.branches][-7:] == ["c", "cpp", "amd64", "aapcs64", "c_types", "cpp_types",
                                                  "c_preprocessor"]  # in install order
# A holder names its ABI, one at most (HasAbi), and a C or C++ term's types are read through it (`C.Domains.Types`):
# among its own dialect's domains, then among its holder's ABI's; the same names read as each ABI fixes them.
X86_64, ARM64 = machine.singleton("Expressions.SystemV.AMD64"), machine.singleton("Expressions.Arm.AAPCS64")
Holder = machine.registered("Expressions.Framework.Expression")


def held(module, abis):
    """A literal of `module`'s, the root of a holder naming the ABIs `abis`."""
    with machine.transaction():
        return machine.Expressions.Framework.Expression(lambda e: e.root(module.Expressions.literal(0)).abi(abis))


amd64, aapcs64, unnamed = held(C, [X86_64]), held(C, [ARM64]), held(C, [])
assert [e.abi for e in amd64.abi] == [X86_64] and list(unnamed.abi) == [] and Validators.Validate(machine)(Holder, amd64) == []
assert len(list(held(C, [X86_64, ARM64]).abi)) == 1  # one at most
x86, arm = C.Domains.Types(Terms.term(amd64)), C.Domains.Types(Terms.term(aapcs64))
assert [(t.kind, t.width, t.signed, t.format, t.name()) for t in map(x86.named, ["bool", "char", "unsigned char", "int8_t", "long double", "_Decimal64"])] == [
    ("bool", 1, False, None, "bool"), ("integer", 8, True, None, "char"), ("integer", 8, False, None, "unsigned char"),
    ("integer", 8, True, None, "int8_t"), ("floating", 80, True, (2, 64, 16383), "long double"),
    ("floating", 64, True, "decimal64", "_Decimal64")]
assert not arm.named("char").signed and arm.named("long double").format == "binary128" and arm.named("wchar_t") == arm.named("unsigned int")
assert x86.named("int").domain is X86_64.domains({"name": "int"}) and arm.named("int").domain is ARM64.domains({"name": "int"})
assert x86.named("int8_t").domain is machine.singleton("Expressions.C.Expressions").domains({"name": "int8_t"})  # C's own
cpp = C.Domains.Types(Terms.term(held(Cpp, [ARM64])), "Cpp.Expressions")  # one ABI serves C and C++
assert cpp.named("std::bfloat16_t").format == (2, 8, 127) and cpp.named("size_t").domain is ARM64.domains({"name": "size_t"})
assert cpp.named("int8_t").domain is machine.singleton("Expressions.Cpp.Expressions").domains({"name": "int8_t"})  # C++'s own
for types, name, message in [
    (C.Domains.Types(Terms.term(unnamed)), "int", "'int' is not a type C defines, and its expression names no ABI"),
    (x86, "my_t", "'my_t' is not a type C or AMD64 defines"), (cpp, "_Float16", "'_Float16' is not a type Cpp or AAPCS64 defines"),
]:
    with raises(ValueError, match=message):
        types.named(name)
assert C.Domains.Types(Terms.term(unnamed)).named("uint8_t").name() == "uint8_t"  # C's own: no ABI needed
bare = Builders.Store()  # the installs, and no define: no domain
for install in MACHINE:
    bare.transaction(install)
with bare.transaction():
    undefined = bare.Expressions.Framework.Expression(lambda e: e.root(C.Expressions.literal(0)).abi([bare.singleton("Expressions.SystemV.AMD64")]))
with raises(ValueError, match="'int' is not a type C or AMD64 defines"):
    C.Domains.Types(Terms.term(undefined)).named("int")
with machine.transaction():
    lone = machine.Expressions.Framework.Term(C.Expressions.literal(0))
with raises(TypeError, match="the value does not belong to a store"):  # a lone term: no holder, no store
    C.Domains.Types(lone).named("int")
# Run again, each define writes nothing; the store round-trips whole into one with the same installs.
snapshot = JSON.ToJSON(machine).Snapshot()
assert [machine.transaction(module.define)[0]["dialect"] for module in DEFINES] == [frames[module]["dialect"] for module in DEFINES]
assert machine.transaction(AMD64.define)[0]["unsigned_long"] is X86_64.domains({"name": "unsigned long"})
assert JSON.ToJSON(machine).Snapshot() == snapshot
again = Builders.Store()
for install in MACHINE:
    again.transaction(install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot
for module, (path, expected) in DEFINED.items():
    assert {str(e.name): (FD.kind_of(e.domain)[:2], FD.properties(e.domain)) for e in again.singleton(path).domains} == {
        name: (kind, properties) for name, (kind, properties, _) in expected.items()}, path

<a id="DOM-20"></a>
## DOM-20 · SystemVerilog's domains are stored: its define writes IEEE 1800's built-in types, by name, of its kinds logic and bit or IEEE 754's, with their properties and, for the 2-state ones, their schemas

In [ ]:
# SystemVerilog's define, run after the installs in a transaction of its own: its domains, IEEE 1800's built-in types,
# owned by its singleton by name, in order, each of its own kinds, `logic` (4-state) and `bit` (2-state), or of IEEE
# 754's, with its parameters and its inferred properties; each 2-state one is related to its schema, an OfAny of the
# scalar it narrows, both ways, and a 4-state one has none, since no basic scalar holds x or z.
hardware = Builders.Store()
for install in [*INSTALLS, SystemVerilog.install]:
    hardware.transaction(install)
SV_PATH = "Expressions.SystemVerilog.Expressions"


def integral(width, signed, least, greatest):
    """An integral domain's properties: its parameters, then its least and greatest known values, and ordered."""
    return {"width": width, "signed": signed, "least": least, "greatest": greatest, "ordered": True}


LOGIC, BIT = ("systemverilog", "logic"), ("systemverilog", "bit")
REAL = {"format": "binary64", "rounding": "roundTiesToEven", "width": 64, "radix": 2, "precision": 53, "emax": 1023,
        "ordered": True}
SHORTREAL = {"format": "binary32", "rounding": "roundTiesToEven", "width": 32, "radix": 2, "precision": 24, "emax": 127,
             "ordered": True}
SYSTEMVERILOG = {  # its domains in order: its kind, its properties, the scalar its schema narrows (None: 4-state, none)
    "logic": (LOGIC, integral(1, False, 0, 1), None), "reg": (LOGIC, integral(1, False, 0, 1), None),
    "integer": (LOGIC, integral(32, True, -2 ** 31, 2 ** 31 - 1), None), "time": (LOGIC, integral(64, False, 0, 2 ** 64 - 1), None),
    "bit": (BIT, integral(1, False, 0, 1), "int"), "byte": (BIT, integral(8, True, -128, 127), "int"),
    "shortint": (BIT, integral(16, True, -2 ** 15, 2 ** 15 - 1), "int"), "int": (BIT, integral(32, True, -2 ** 31, 2 ** 31 - 1), "int"),
    "longint": (BIT, integral(64, True, -2 ** 63, 2 ** 63 - 1), "int"),
    "real": (("ieee754", "ieee754"), REAL, "float"), "shortreal": (("ieee754", "ieee754"), SHORTREAL, "float"),
}
assert SystemVerilog.Expressions.define is SystemVerilog.define  # its package exports its module's
frame = hardware.transaction(SystemVerilog.define)[0]
sv_dialect = hardware.singleton(SV_PATH)
assert frame["dialect"] is sv_dialect and [str(e.name) for e in sv_dialect.domains] == list(SYSTEMVERILOG)
HardwareDomain = hardware.registered("Expressions.Framework.Domain")
for name, (kind, properties, scalar) in SYSTEMVERILOG.items():
    domain = sv_dialect.domains({"name": name})
    assert frame[name] is domain and Validators.Validate(hardware)(HardwareDomain, domain) == [], name
    assert FD.kind_of(domain)[:2] == kind and FD.properties(domain) == properties and FD.name_of(domain) == name
    if scalar is None:  # 4-state: no basic scalar holds x or z, so no schema
        assert list(domain.schema) == []
        with raises(LookupError, match=f"no schema registered as '{SV_PATH}.{name}'"):
            hardware.registered(f"{SV_PATH}.{name}")
        continue
    schema = hardware.registered(f"{SV_PATH}.{name}")
    assert schema.schemas == (hardware.registered(f"Schemas.Of{scalar.capitalize()}"),) and schema.validate() == []
    assert [e.schema for e in domain.schema] == [schema]  # HasDomain, both ways
    read = [e.domain for e in Builders.OfAny.Builder(schema, False, hardware).domain]
    assert [(FD.name_of(d), FD.kind_of(d)[:2], FD.properties(d)) for d in read] == [(name, kind, properties)]
# Its kinds, each its own record schema, their parameters, then their inferred properties; `reg` is `logic`'s equal,
# its own domain, not a synonym; `real` and `shortreal` hold IEEE 754's parameters of their formats.
for tag, record in [("logic", "OfLogic"), ("bit", "OfBit")]:
    assert list(hardware.registered(f"{SV_PATH}.{record}").properties) == ["width", "signed", "least", "greatest", "ordered"]
    assert FD.kind_of(sv_dialect.domains({"name": tag}))[2].as_schema() is hardware.registered(f"{SV_PATH}.{record}")
logic, reg = sv_dialect.domains({"name": "logic"}), sv_dialect.domains({"name": "reg"})
assert reg is not logic and FD.properties(reg) == FD.properties(logic)
assert FD.properties(domains["binary64"]) == REAL and FD.properties(domains["binary32"]) == SHORTREAL  # IEEE 754's own
# Each is a type SystemVerilog's evaluation, inference and validation read by name, as its working value: its width,
# signedness, states (by its kind) and format, labelled with its name.
assert [(t.kind, t.width, t.signed, t.states, t.format, t.name()) for t in (SystemVerilog.Domains.named(hardware, name) for name in SYSTEMVERILOG)] == [
    ("integral", 1, False, 4, None, "logic"), ("integral", 1, False, 4, None, "reg"), ("integral", 32, True, 4, None, "integer"),
    ("integral", 64, False, 4, None, "time"), ("integral", 1, False, 2, None, "bit"), ("integral", 8, True, 2, None, "byte"),
    ("integral", 16, True, 2, None, "shortint"), ("integral", 32, True, 2, None, "int"), ("integral", 64, True, 2, None, "longint"),
    ("real", 64, True, 2, "binary64", "real"), ("real", 32, True, 2, "binary32", "shortreal")]
# Run again, the define writes nothing; the store round-trips whole into one with the same installs.
snapshot = JSON.ToJSON(hardware).Snapshot()
assert hardware.transaction(SystemVerilog.define)[0]["int"] is sv_dialect.domains({"name": "int"})
assert JSON.ToJSON(hardware).Snapshot() == snapshot
again = Builders.Store()
for install in [*INSTALLS, SystemVerilog.install]:
    again.transaction(install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot
assert {str(e.name): (FD.kind_of(e.domain)[:2], FD.properties(e.domain)) for e in again.singleton(SV_PATH).domains} == {
    name: (kind, properties) for name, (kind, properties, _) in SYSTEMVERILOG.items()}

<a id="DOM-21"></a>
## DOM-21 · Python3's, TypeScript5's, MATLAB's and Excel's domains are stored: each define writes its dialect's, by name, with their kinds, properties and schemas

In [ ]:
# Python3's, TypeScript5's, MATLAB's and Excel's defines, each run after the installs in a transaction of its own:
# each writes its domains, owned by its singleton by name, in order, each named as its language names its type, of a
# standard dialect's kind or its own, with its parameters and its inferred properties; each but a token's is related
# to its schema, an OfAny of the scalar it narrows, both ways.
stored = Builders.Store()
for install in [*INSTALLS, Python3.install, Matlab.install, Excel.install, TypeScript5.install, PythonTypes.install,
                PythonPatterns.install, TypeScriptTypes.install, TypeScriptPatterns.install, PythonRegExp.install,
                TypeScriptRegExp.install]:  # each after its family's
    stored.transaction(install)
ORDERED, UNORDERED = {"ordered": True}, {"ordered": False}
UNBOUNDED = {"signed": True, "overflow": "raise", "ordered": True}  # 11404's integer: no width, no least or greatest
BINARY64 = {"format": "binary64", "rounding": "roundTiesToEven", "width": 64, "radix": 2, "precision": 53, "emax": 1023,
            "ordered": True}
EXCEL_ERRORS = ("#NULL!", "#DIV/0!", "#VALUE!", "#REF!", "#NAME?", "#NUM!", "#N/A", "#GETTING_DATA",  # ERROR.TYPE's
                "#SPILL!", "#CALC!", "#FIELD!", "#BLOCKED!", "#CONNECT!", "#BUSY!", "#UNKNOWN!")  # then added since
# Populated by hand, a standard kind's domain is as its standard dialect's own: binary64, integer, boolean, octetstring.
assert [BINARY64, UNBOUNDED, UNORDERED, ORDERED] == [
    FD.properties(domains[name]) for name in ("binary64", "integer", "boolean", "octetstring")]
STORED = {  # each dialect's path, and its domains in order: its kind, its properties, the scalar its schema narrows
    Python3: ("Expressions.Python.Python3", {
        "bool": (("isoiec11404", "boolean"), UNORDERED, "bool"), "int": (("isoiec11404", "integer"), UNBOUNDED, "int"),
        "float": (("ieee754", "ieee754"), BINARY64, "float"), "str": (("python3", "str"), ORDERED, "str"),
        "bytes": (("isoiec11404", "octetstring"), ORDERED, "bytes"),
        "NoneType": (("python3", "nonetype"), UNORDERED, None), "ellipsis": (("python3", "ellipsis"), UNORDERED, None)}),
    TypeScript5: ("Expressions.TypeScript.TypeScript5", {
        "boolean": (("isoiec11404", "boolean"), UNORDERED, "bool"), "number": (("ieee754", "ieee754"), BINARY64, "float"),
        "bigint": (("isoiec11404", "integer"), UNBOUNDED, "int"), "string": (("typescript5", "string"), ORDERED, "str"),
        "null": (("typescript5", "null"), UNORDERED, None)}),
    Matlab: ("Expressions.MATLAB.Expressions", {
        "double": (("ieee754", "ieee754"), BINARY64, "float"), "logical": (("isoiec11404", "boolean"), UNORDERED, "bool"),
        "string": (("matlab", "string"), ORDERED, "str")}),
    Excel: ("Expressions.Excel.Expressions", {
        "number": (("ieee754", "ieee754"), BINARY64, "float"), "logical": (("isoiec11404", "boolean"), UNORDERED, "bool"),
        "text": (("excel", "text"), ORDERED, "str"),
        "error": (("excel", "error"), {"members": EXCEL_ERRORS, "ordered": False}, "str")}),  # an error value: its code
}
StoredDomain = stored.registered("Expressions.Framework.Domain")
for module, (path, expected) in STORED.items():
    assert module.Expressions.define is module.define  # its package exports its module's
    frame = stored.transaction(module.define)[0]
    dialect = stored.singleton(path)
    assert frame["dialect"] is dialect and [str(e.name) for e in dialect.domains] == list(expected)
    for name, (kind, properties, scalar) in expected.items():
        domain = dialect.domains({"name": name})
        assert frame[name] is domain and Validators.Validate(stored)(StoredDomain, domain) == [], (path, name)
        assert FD.kind_of(domain)[:2] == kind and FD.properties(domain) == properties and FD.name_of(domain) == name
        if scalar is None:  # a token: no basic scalar holds its value, so no schema
            assert list(domain.schema) == []
            with raises(LookupError, match=f"no schema registered as '{path}.{name}'"):
                stored.registered(f"{path}.{name}")
            continue
        schema = stored.registered(f"{path}.{name}")
        assert schema.schemas == (stored.registered(f"Schemas.Of{scalar.capitalize()}"),) and schema.validate() == []
        assert [e.schema for e in domain.schema] == [schema]  # HasDomain, both ways
        read = [e.domain for e in Builders.OfAny.Builder(schema, False, stored).domain]
        assert [(FD.name_of(d), FD.kind_of(d)[:2], FD.properties(d)) for d in read] == [(name, kind, properties)]
# The new kinds' records are their dialects' own.
assert FD.kind_of(stored.singleton("Expressions.MATLAB.Expressions").domains({"name": "string"}))[2].as_schema() is (
    stored.registered("Expressions.MATLAB.Expressions.OfString"))
assert FD.kind_of(stored.singleton("Expressions.Excel.Expressions").domains({"name": "text"}))[2].as_schema() is (
    stored.registered("Expressions.Excel.Expressions.OfText"))
# Run again, each define writes nothing; the store round-trips whole into one with the same installs.
snapshot = JSON.ToJSON(stored).Snapshot()
for module in STORED:
    stored.transaction(module.define)
assert JSON.ToJSON(stored).Snapshot() == snapshot
again = Builders.Store()
for install in [*INSTALLS, Python3.install, Matlab.install, Excel.install, TypeScript5.install, PythonTypes.install,
                PythonPatterns.install, TypeScriptTypes.install, TypeScriptPatterns.install, PythonRegExp.install,
                TypeScriptRegExp.install]:  # the same installs
    again.transaction(install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot
assert {path: {str(e.name): (FD.kind_of(e.domain)[:2], FD.properties(e.domain)) for e in again.singleton(path).domains}
        for path, _ in STORED.values()} == {
    path: {name: (kind, properties) for name, (kind, properties, _) in expected.items()} for path, expected in STORED.values()}